In [1]:
import numpy as np
from PIL import Image

TARGET_IMAGE = "test_files/blank_recovered.png"

img = Image.open(TARGET_IMAGE).convert("RGB")
arr = np.array(img)

# 1. Test different extraction channels: Interleaved RGB vs isolated Red/Green/Blue
extraction_modes = {
    "Interleaved RGB": (arr & 1).flatten(),
    "Red Channel LSB": (arr[:, :, 0] & 1).flatten(),
    "Green Channel LSB": (arr[:, :, 1] & 1).flatten(),
    "Blue Channel LSB": (arr[:, :, 2] & 1).flatten(),
}

print(f"{'EXTRACTION MODE':<20} {'HEX (First 16 Bytes)':<35} {'ASCII PREVIEW'}")
print("-" * 75)

for mode_name, bits in extraction_modes.items():
    raw_bytes = np.packbits(bits)
    first_16 = bytes(raw_bytes[:16])
    hex_repr = first_16.hex(" ")
    ascii_repr = "".join(chr(b) if 32 <= b < 127 else "." for b in first_16)
    
    print(f"{mode_name:<20} {hex_repr:<35} {ascii_repr}")

# Quick scan of the first 256 bytes from Interleaved RGB for ASCII blocks
raw_interleaved = bytes(np.packbits(extraction_modes["Interleaved RGB"])[:256])

KNOWN_HEADERS = {
    b"PK\x03\x04": "ZIP archive / Office doc",
    b"7z\xbc\xaf\x27\x1c": "7-Zip archive",
    b"\x7fELF": "Linux ELF binary",
    b"MZ": "Windows Executable (PE)",
    b"%PDF": "PDF document",
    b"\x1f\x8b": "GZIP compressed data",
}

detected_magic = False
for sig, label in KNOWN_HEADERS.items():
    if raw_interleaved.startswith(sig):
        print(f"\n[!] Confirmed payload signature detected: {label}")
        detected_magic = True
        break

if not detected_magic:
    print("\nResult: No standard container signatures located at offset 0.")

EXTRACTION MODE      HEX (First 16 Bytes)                ASCII PREVIEW
---------------------------------------------------------------------------
Interleaved RGB      6d b6 db 6d b6 db 6d b6 db 6d b6 db 6d b6 db 6d m..m..m..m..m..m
Red Channel LSB      00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 ................
Green Channel LSB    ff ff ff ff ff ff ff ff ff ff ff ff ff ff ff ff ................
Blue Channel LSB     ff ff ff ff ff ff ff ff ff ff ff ff ff ff ff ff ................

Result: No standard container signatures located at offset 0.
